# Hashing Files
Hashing a file proves it hasn't changed, for example after a download. Files can be huge, so we need a way to hash them without loading the whole file into memory at once.

1. Hash a small file the simple way
2. Hash in pieces with .update()
3. Hash a file in chunks (the proper way)
4. The shortcut: hashlib.file_digest()
5. Verify a file against a published hash

## Step 1: Hash a small file the simple way
First we make a test file, then read all of it and hash it. Files must be opened in binary mode (`"rb"`) so we get bytes, not text.

In [2]:
import hashlib
import os

# Make a test file
with open("sample.txt", "w", encoding="utf-8") as f:
    f.write("Khemshield security advisory\n" * 1000)

print("File size:", os.path.getsize("sample.txt"), "bytes")

# Read the whole file at once and hash it
with open("sample.txt", "rb") as f:
    data = f.read()

print(type(data))
print(hashlib.sha256(data).hexdigest())

File size: 30000 bytes
<class 'bytes'>
726bbc3cdfc513d50e96b4caa3011bf68a8617ba84cc0183622176c0a9489130


## Step 1b: Why newline="" matters
On Windows, writing a file in text mode quietly turns every new line (`\n`) into two characters (`\r\n`). On Mac and Linux it doesn't. So the same code can make different bytes on different computers, and different bytes mean a different hash.

`newline=""` tells Python not to change the new lines, so the file is the same everywhere.

Here we write the same text both ways and compare.

In [3]:
text = "Khemshield security advisory\n" * 1000

# Default: the operating system may change \n
with open("default.txt", "w", encoding="utf-8") as f:
    f.write(text)

# newline="": write \n exactly as it is
with open("fixed.txt", "w", encoding="utf-8", newline="") as f:
    f.write(text)

for name in ["default.txt", "fixed.txt"]:
    with open(name, "rb") as f:
        data = f.read()
        
    print(name)
    print("  Size:", os.path.getsize(name), "bytes")
    print("  End of first line:", data[26:31])
    print("  SHA-256:", hashlib.sha256(data).hexdigest())

# Clean up the test files
os.remove("default.txt")
os.remove("fixed.txt")

default.txt
  Size: 30000 bytes
  End of first line: b'ry\r\nK'
  SHA-256: 726bbc3cdfc513d50e96b4caa3011bf68a8617ba84cc0183622176c0a9489130
fixed.txt
  Size: 29000 bytes
  End of first line: b'ry\nKh'
  SHA-256: 7272697f47d65ec74f6d59c7c2e3a453db0c2211140bfb1f2aa5089b684a35a3


## Step 2: Hash in pieces with .update()
A hash object can be fed data a piece at a time with `.update()`. The final hash is the same as hashing everything at once, as long as the pieces join up to the same bytes in the same order.

In [4]:
import hashlib

# All at once
print(hashlib.sha256(b"Khemshield security advisory").hexdigest())

# In pieces
h = hashlib.sha256()          # start with an empty hash object
h.update(b"Khemshield ")
h.update(b"security ")
h.update(b"advisory")
print(h.hexdigest())

# Where you cut the pieces doesn't matter, only the joined bytes
h2 = hashlib.sha256()
h2.update(b"Khem")
h2.update(b"shield security advis")
h2.update(b"ory")
print(h2.hexdigest())

# But the order does matter
h3 = hashlib.sha256()
h3.update(b"advisory")
h3.update(b"Khemshield security ")
print(h3.hexdigest())

3e44a490256a1a22fd03573db189e96df8f389ef6378710c67f7436f70cb836c
3e44a490256a1a22fd03573db189e96df8f389ef6378710c67f7436f70cb836c
3e44a490256a1a22fd03573db189e96df8f389ef6378710c67f7436f70cb836c
697e2e540f4c00ee339a2c03d504a62e5e81a66c18a0748974f1821032cbd9f4


The text was the same, but the bytes were not, so the hashes were different. When a file's hash needs to match on every computer, control the bytes exactly: use `newline=""` when writing text, or write in binary mode (`"wb"`).

## Step 3: Hash a file in chunks
Now we turn the chunk method into a reusable function. It reads the file a piece at a time, so memory stays small however big the file is.

This uses the **walrus operator** `:=`, which saves a value into a variable *and* uses it in the same line. It needs Python 3.8 or newer.

In [5]:
def hash_file(path: str, algorithm: str = "sha256", chunk_size: int = 65536) -> str:
    h = hashlib.new(algorithm)
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

# Compare with the simple way from Step 1
with open("sample.txt", "rb") as f:
    simple = hashlib.sha256(f.read()).hexdigest()

chunked = hash_file("sample.txt")

print("Simple: ", simple)
print("Chunked:", chunked)
print("Same?  ", simple == chunked)

Simple:  726bbc3cdfc513d50e96b4caa3011bf68a8617ba84cc0183622176c0a9489130
Chunked: 726bbc3cdfc513d50e96b4caa3011bf68a8617ba84cc0183622176c0a9489130
Same?   True


## Step 4: The shortcut: hashlib.file_digest()
Python 3.11 added `hashlib.file_digest()`, which does the chunked reading for us. We give it an open file and the name of the algorithm, and it returns a hash object.

In [6]:
import sys

print("Python version:", sys.version.split()[0])

with open("sample.txt", "rb") as f:
    h = hashlib.file_digest(f, "sha256")

print(h.hexdigest())
print(h.hexdigest() == hash_file("sample.txt"))   # same as our Step 3 function

Python version: 3.12.13
726bbc3cdfc513d50e96b4caa3011bf68a8617ba84cc0183622176c0a9489130
True


### Things to know
- It needs **Python 3.11 or newer**. On older versions you get an `AttributeError`, so use the Step 3 function instead.
- The file must be opened in binary mode (`"rb"`).
- It returns a hash object, not a string, so you still call `.hexdigest()` or `.digest()`.
- You can pass any algorithm name, for example `"sha512"` or `"sha3_256"`.

So why learn Step 3 at all? Because it shows what the shortcut does inside, it works on older Python versions, and the same pattern is needed for things that have no shortcut, like an HMAC of a big file.

## Step 5: Verify a file against a published hash
Publishers often list a SHA-256 hash next to a download. After downloading, we hash the file ourselves and compare. If it matches, the file arrived exactly as published. If even one byte changed, it won't match.

Here we pretend to be the publisher first, then check the original file and a tampered copy.

In [7]:
import hmac

def verify_file(path: str, expected: str, algorithm: str = "sha256") -> bool:
    actual = hash_file(path, algorithm)
    return hmac.compare_digest(actual, expected.strip().lower())

# The publisher hashes the file and posts the result online
published_hash = hash_file("sample.txt")
print("Published:", published_hash)

# We download it and check it
print("Original file OK?", verify_file("sample.txt", published_hash))

Published: 726bbc3cdfc513d50e96b4caa3011bf68a8617ba84cc0183622176c0a9489130
Original file OK? True


### Tampering with one byte
We make a copy of the file and change a single byte, then verify again.

In [8]:
with open("sample.txt", "rb") as f:
    data = bytearray(f.read())   # bytearray so we can change it

data[0] = ord("X")               # change the first byte only ("K" becomes "X")

with open("tampered.txt", "wb") as f:
    f.write(data)

print("Same size?    ", os.path.getsize("sample.txt") == os.path.getsize("tampered.txt"))
print("Tampered OK?  ", verify_file("tampered.txt", published_hash))
print("Tampered hash:", hash_file("tampered.txt"))

os.remove("tampered.txt")

Same size?     True
Tampered OK?   False
Tampered hash: 466c86b49f1c2de99b1c9e74b87085beec78f5a78636c2d205cdb4c16f63b22f


### Checksum files
Publishers often list many hashes in one file, called `SHA256SUMS`, with a hash and a file name on each line. We can read the hash for our file from it.

In [ ]:
# A pretend SHA256SUMS file from a publisher
sums_text = f"{published_hash} sample.txt\n{'a' * 64}  other-file.zip\n"

def expected_from_sums(sums: str, filename: str) -> str:
    for line in sums.splitlines():
        file_hash, name = line.split()
        if name == filename:
            return file_hash
    raise ValueError(f"{filename} is not listed")

expected = expected_from_sums(sums_text, "sample.txt")
print("From SHA256SUMS:", expected)
print("Verified?       ", verify_file("sample.txt", expected))

From SHA256SUMS: 726bbc3cdfc513d50e96b4caa3011bf68a8617ba84cc0183622176c0a9489130
Verified?        True


### Checking a hash without Python
On Windows, you can hash a file from the terminal:
- Command Prompt: `certutil -hashfile sample.txt SHA256`
- PowerShell: `Get-FileHash sample.txt`

On Mac or Linux: `shasum -a 256 sample.txt` or `sha256sum sample.txt`.

These should give the same hash as our function (Windows shows it in capitals, which is why we use `.lower()` when comparing).

### The limit of file hashes
A matching hash only proves the file matches the hash you were given. If an attacker can change both the download **and** the hash on the website, the check still passes. That's why serious publishers also **sign** their hash lists, which is what digital signatures (Topic 2) are for.

## Summary
- Always open files in binary mode (`"rb"`) when hashing.
- On Windows, text mode changes `\n` to `\r\n`. Use `newline=""` (or `"wb"`) so the bytes are the same everywhere.
- Big files should be hashed in chunks with `.update()` so memory stays small. Chunk size doesn't change the hash, only the number of pieces.
- Every byte must go in, in order. Never skip parts or strip lines.
- `hashlib.file_digest()` (Python 3.11+) does the chunking for you.
- To verify a download, hash it yourself and compare with the published hash. One changed byte means no match.
- A file hash covers only the content, not the name or date.
- A hash proves the file matches what was published, not who published it. For that we need signatures.